In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from ase.build import bulk, make_supercell
from ase.neighborlist import neighbor_list
from tqdm.notebook import tqdm
import molpot as mpot
from molpot import alias

## data prepare

In [2]:
def ase2frame(frame, device="cpu"):

    _frame = mpot.Frame()
    _frame[alias.n_atoms] = len(frame)
    _frame[alias.xyz] = torch.from_numpy(frame.get_positions())
    _frame[alias.cell] = torch.from_numpy(frame.get_cell().array)
    _frame[alias.pbc] = torch.from_numpy(frame.get_pbc())

    _frame.to(device)
    return _frame

In [3]:
frame = bulk('Si', 'diamond', a=4, cubic=True)
aa = torch.arange(1, 6)
Ps = torch.cartesian_prod(aa,aa,aa)
Ps = Ps[torch.sort(Ps.sum(dim=1)).indices].to(torch.long).numpy()
frames = []
n_atoms = []
for P in Ps:
    frames.append(make_supercell(frame, np.diag(P)))
    n_atoms.append(len(frames[-1]))
n_atoms = np.array(n_atoms)

In [4]:
from torch_nl import compute_neighborlist, compute_neighborlist_n2, ase2data
from torch_nl.timer import timeit

In [5]:
cutoff = 4

In [6]:
tag = "ASE"
datas = []
for frame in tqdm(frames):
    timing = timeit(neighbor_list, ['ijS', frame, cutoff], tag=tag, warmup=1, nit=50)
    data = timing.dumps()
    i,j,S = neighbor_list('ijS', frame, cutoff)
    n_neighbor = np.bincount(i).mean()
    data.update(n_atom=len(frame), n_neighbor_per_atom_avg=int(n_neighbor))
    data.pop('samples')
    datas.append(data)

  0%|          | 0/125 [00:00<?, ?it/s]

In [7]:
tags = [
    # "torch_nl O(n^2) CPU", 
    "torch_nl O(n^2) GPU", 
    "torch_nl O(n) CPU", 
    "torch_nl O(n) GPU"
]
for tag in tqdm(tags):
    if "CPU" in tag:
        device = 'cpu'
    elif "GPU" in tag:
        device = 'cuda'
        
    if 'O(n^2)' in tag:
        nl_func = compute_neighborlist_n2
    elif 'O(n)' in tag:
        nl_func = compute_neighborlist

    for frame in tqdm(frames):
        pos, cell, pbc, batch, n_atoms = ase2data([frame], device=device)
        timing = timeit(nl_func, [cutoff, pos, cell, pbc, batch], tag=tag, warmup=10, nit=50)
        data = timing.dumps()
        data.pop('samples')
        mapping, mapping_batch, shifts_idx = nl_func(cutoff, pos, cell, pbc, batch)
        n_neighbor = np.bincount(mapping[0].cpu().numpy()).mean()
        data.update(n_atom=len(frame), n_neighbor_per_atom_avg=int(n_neighbor))
        datas.append(data)

  0%|          | 0/3 [00:00<?, ?it/s]

NameError: name 'frames' is not defined

In [ ]:
df = pd.DataFrame(datas)
df

In [ ]:
with sns.plotting_context("notebook"):
    plt.title("Compute neighborlist: diamond structure")
    sns.lmplot(data=df, x='n_atom', y='mean', hue='tag',fit_reg=False)
    plt.xlabel('timing []')
    plt.savefig('diamond_benchmark.png', dpi=300, bbox_inches='tight')